<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/04_memoria_corto_plazo.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap04) · 🧭 [Teoría del capítulo](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap04-memoria-corto-plazo.html) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 3](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/03_runtime.ipynb) · ➡️ [Cap. 5](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/05_memoria_largo_plazo.ipynb)

# Capítulo 4 — El agente que olvida tu nombre

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Laura: *«Me llamo Laura»*. Un mensaje después: *«¿Cómo me llamo?»*. El agente no lo sabe.

| | |
|---|---|
| **🧠 Componente agéntico** | Memoria de trabajo y a corto plazo |
| **La necesidad** | Que el agente recuerde la conversación en curso |
| **Lo que construimos** | **AgentCore Memory** (corto plazo) con el *session manager* de Strands |
| **🏠 En local** | ✅ Funciona sin que lo notes: en el capítulo 2 el objeto `Agent` ya guardaba los mensajes en la RAM del proceso, que vive mientras tengas el notebook abierto. |
| **☁️ Por qué AWS al salir a Internet** | En Runtime, la RAM de una sesión es la de su microVM: se pierde a los 15 minutos sin actividad, al redesplegar o a las 8 horas. En un servidor clásico con réplicas pasa lo mismo cuando el cliente cae en otra. Memory guarda la conversación **fuera del cómputo**: cualquier microVM nueva de esa sesión la recupera. |
| **En el ordenador sería…** | La **RAM**: se puede escribir, pero es volátil. Hay que conservar lo que pasa mientras dura la sesión |
| **Requisitos del caso** | Base de RF-05 |
| **Prerrequisito · duración** | Capítulo 3 · 45 min |
| **⚠️ Dónde se rompe la analogía** | La RAM se lee sin coste; la ventana de contexto **se reprocesa entera en cada llamada y se paga cada vez**. De ahí el taxímetro. |

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y la AgentCore CLI 0.31.1 y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=4)

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(4)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(4)
# ponerse_al_dia(4)   # ← descomenta si llegas tarde o algo se rompió

## 1. Experimento: la tentación de guardar la conversación en la microVM

En el capítulo 3 vimos que la microVM de una sesión **conserva su estado** entre mensajes. La tentación es obvia: guardar el `Agent` de cada sesión en un diccionario y reutilizarlo.

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models import BedrockModel

from herramientas import HERRAMIENTAS
from prompts import SYSTEM_PROMPT
from respuesta import respuesta
from settings import MODEL_ID, REGION

app = BedrockAgentCoreApp()

# ⚠️ EXPERIMENTO: el historial de cada sesión se guarda en la RAM de su microVM.
AGENTES: dict[str, Agent] = {}


@app.entrypoint
def invoke(payload, context):
    agent = AGENTES.get(context.session_id)
    if agent is None:
        agent = AGENTES[context.session_id] = Agent(
            model=BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.2),
            system_prompt=SYSTEM_PROMPT,
            tools=HERRAMIENTAS,
            callback_handler=None,
        )
    return respuesta(agent(payload.get("prompt", "")))


if __name__ == "__main__":
    app.run()

In [ ]:
cli("agentcore deploy --yes")

> ✋ **Predice antes de ejecutar.** Misma sesión, dos mensajes: ¿recordará ahora el nombre? Después **paramos la microVM** de esa sesión (es lo que pasa tras 15 minutos sin actividad, al redesplegar o a las 8 horas) y volvemos a preguntar con el mismo `session_id`. ¿Lo recordará?

In [ ]:
sid = new_session_id()
invocar("Hola, me llamo Laura.", session_id=sid);  a = ultima_microvm()
invocar("¿Cómo me llamo?", session_id=sid);        b = ultima_microvm(); antes = ultima_respuesta()
parar_sesion(sid)                                  # la microVM de la sesión se apaga
invocar("¿Cómo me llamo?", session_id=sid);        c = ultima_microvm(); despues = ultima_respuesta()   # mismo id

In [ ]:
print(f"Mensajes 1 y 2: misma microVM {a['id'] == b['id']} · ¿recuerda el nombre? {'laura' in antes.lower()}")
print(f"Tras parar la sesión: microVM nueva {c['id'] != a['id']} · ¿recuerda el nombre? {'laura' in despues.lower()}")

Mientras la microVM vive, el diccionario funciona: recuerda el nombre. Cuando la microVM se apaga, se lleva la conversación con ella, aunque el `session_id` sea el mismo. Y Laura no vuelve en 15 minutos: vuelve **mañana**, o desde la app en lugar de la web, o después de que hayamos desplegado una versión nueva. En un servidor clásico con varias réplicas pasaría lo mismo cada vez que el balanceador la mandara a otra.

> 🧭 **¿Por qué ha pasado?** Abre la teoría: [**La RAM del agente**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap04-memoria-corto-plazo.html). Repite la conversación de Laura en tu máquina, en una microVM de Runtime y con Memory, y apaga la microVM a mitad. Vuelve cuando sepas responder a esta pregunta: **¿dónde tiene que vivir la conversación para que sobreviva a la microVM que la atendió?**

**En dos líneas:** la *memoria de trabajo* es lo que el modelo ve en esta llamada; la *memoria a corto plazo* es el historial que hay que guardar en algún sitio y reenviar en cada llamada. La RAM de una microVM sirve mientras vive, pero se apaga sin avisar (inactividad, despliegues, límite de 8 horas): el historial tiene que vivir en un servicio, **fuera del cómputo**, y el agente debe poder retomarlo desde cualquier microVM.

## 2. Crear la memoria con la CLI

Se crea ya con estrategias de largo plazo (las usará el capítulo 5). Después, `main.py` vuelve a crear un `Agent` nuevo en cada invocación: el historial lo carga el *session manager* desde Memory. Así la CLI gestiona también los permisos del Runtime sobre la memoria.

In [ ]:
cli('agentcore add memory --name NovaMemory --strategies SEMANTIC,SUMMARIZATION,USER_PREFERENCE')
cli('agentcore deploy --yes')

In [ ]:
mid = buscar_memoria("NovaMemory")
if mid:
    state_set(MEMORY_ID=mid)
# Si no la encuentra: state_set(MEMORY_ID="<ID de agentcore status>")

## 3. Código

In [ ]:
fijar_dependencia(VERSIONES["bedrock-agentcore"])
write_settings()

In [ ]:
%%writefile NovaAssist/app/NovaAgent/memoria.py
"""Memoria de AgentCore para Strands: corto plazo (eventos) y largo plazo (recuperación por namespaces)."""
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

from settings import LTM_NAMESPACES, MEMORY_ID, REGION


def retrieval_config() -> dict:
    """Vacío en el capítulo 4 (solo corto plazo); en el capítulo 5 se rellena desde LTM_NAMESPACES."""
    return {ns: RetrievalConfig(top_k=v[0], relevance_score=v[1]) for ns, v in LTM_NAMESPACES.items()}


def crear_session_manager(actor_id: str, session_id: str, retrieval: dict | None = None):
    kwargs = {"memory_id": MEMORY_ID, "session_id": session_id, "actor_id": actor_id}
    if retrieval:
        kwargs["retrieval_config"] = retrieval
    return AgentCoreMemorySessionManager(AgentCoreMemoryConfig(**kwargs), region_name=REGION)

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models import BedrockModel

from herramientas import HERRAMIENTAS
from memoria import crear_session_manager, retrieval_config
from prompts import SYSTEM_PROMPT
from respuesta import respuesta
from settings import MODEL_ID, REGION

app = BedrockAgentCoreApp()


@app.entrypoint
def invoke(payload, context):
    actor_id = payload.get("cliente_id", "anonimo")  # ⚠️ provisional: en el capítulo 9 saldrá del token
    # El Agent sigue siendo nuevo en cada invocación: el historial lo carga el session manager desde Memory.
    with crear_session_manager(actor_id, context.session_id, retrieval_config()) as sm:
        agent = Agent(
            model=BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.2),
            system_prompt=SYSTEM_PROMPT,
            tools=HERRAMIENTAS,
            session_manager=sm,
            callback_handler=None,
        )
        result = agent(payload.get("prompt", ""))
    return respuesta(result)


if __name__ == "__main__":
    app.run()

## 4. Prueba local: misma sesión frente a sesión nueva

In [ ]:
sid = new_session_id()
probar_local("Me llamo Laura", session_id=sid)
probar_local("¿Cómo me llamo?", session_id=sid)          # misma sesión → lo recuerda
probar_local("¿Cómo me llamo?", session_id=new_session_id())   # sesión nueva → no lo sabe

## 5. Desplegar y repetir el experimento

> ✋ **Predice antes de ejecutar.** Mismo experimento que en la sección 1: dos mensajes, paramos la microVM y volvemos a preguntar. ¿Qué cambia ahora?

In [ ]:
cli('agentcore deploy --yes')

In [ ]:
sid = new_session_id()
invocar("Hola, me llamo Laura.", session_id=sid);  a = ultima_microvm()
parar_sesion(sid)                                  # la microVM se apaga…
invocar("¿Cómo me llamo?", session_id=sid);        b = ultima_microvm()
print(f"microVM nueva: {a['id'] != b['id']} · ¿recuerda el nombre? {'laura' in ultima_respuesta().lower()}")

Otra microVM, y aun así recuerda: la conversación ya no vive en el cómputo, sino en **Memory**. **Verificación adicional:** en la consola de AgentCore, abre **Memory → NovaMemory** y localiza los eventos de la sesión.

> `context.session_id` es el `runtimeSessionId`. Si tu versión del SDK lo expone con otro nombre, consulta la referencia de `RequestContext`. Si aparece `AccessDenied` sobre Memory en el Runtime, revisa los permisos del rol de ejecución.

## Para pensar

- ¿Qué ocurre si dos clientes comparten `actor_id`?

## 🔎 Lo que acabamos de descubrir

Laura vuelve **mañana**, en una sesión nueva, y el agente no recuerda que prefiere WhatsApp. Lo que guardamos dura lo que dura la sesión. Además, el `cliente_id` sigue llegando en el payload: cualquiera puede escribir el de otro.

In [ ]:
deuda("D4", "abierta", 4, "cliente_id en el payload")
cuaderno()
taximetro_cerrar(4)
guardar_partida(4)

📚 **Para repasar más adelante:** la teoría de este capítulo está en [`teoria/cap04-memoria-corto-plazo.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap04-memoria-corto-plazo.html) y el mapa completo en [`teoria/index.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). La referencia por componente es la Parte I del documento.

---
➡️ **Siguiente capítulo:** `05_memoria_largo_plazo.ipynb` — *El agente que no te reconoce mañana*

---
**Siguiente:** ➡️ [Capítulo 5 · El agente que no te reconoce mañana](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/05_memoria_largo_plazo.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap05). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.